In [ ]:
import numpy as np
import h5py
import hdf5plugin
import matplotlib.pyplot as plt
from matplotlib import colors
from pathlib import Path
from collections import OrderedDict
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.animation import FuncAnimation
from collections import OrderedDict
from IPython.display import HTML,display

In [ ]:
def animate_image_subsets(data_dict, key_ranges=None, title_dict=None, interval=200, save_path=None):
    """
    Generates and displays separate animations for each specified range of keys.
    
    Parameters:
    -----------
    data_dict : dict or OrderedDict
        Your dictionary containing {float_key: image_array}
    key_ranges : list of tuples
        A list of inclusive (min, max) bounds. Each tuple creates its own animation.
        Example: [(126, 132), (136, 147)]
    title_dict : dict, optional
        Your dictionary containing {float_key: title_string}. If None, defaults to the float key.
    interval : int, default 200
        Time between frames in milliseconds.
    save_path : str, optional
        Base filename for saving. The range bounds will be automatically appended.
    """
    animation_widgets = []
    if not key_ranges:
        key_ranges = [(min(data_dict.keys()), max(data_dict.keys()))]
    for start, end in key_ranges:
        # 1. Filter items belonging strictly to this specific subset range
        subset_items = [
            (k, v) for k, v in data_dict.items() 
            if start <= k <= end
        ]
        # Explicitly sort the subset by its float keys
        subset_items.sort(key=lambda item: item[0])
        
        if not subset_items:
            print(f"Skipping range ({start} to {end}): No matching keys found.")
            continue
            
        keys = [item[0] for item in subset_items]
        frames = [item[1] for item in subset_items]
        
        # 2. Initialize a fresh figure and axis for this specific animation sequence
        fig, ax = plt.subplots()
        ax.axis('off')
        
        im = ax.imshow(frames[0])
        
        # Determine the initial title string based on whether title_dict was provided
        initial_title = title_dict[keys[0]] if title_dict else f"Key: {keys[0]:.4f}"
        ax.set_title(initial_title)
        
        # 3. Define the frame update logic 
        def update(frame_index, keys=keys, frames=frames, im=im, ax=ax):
            im.set_data(frames[frame_index])
            current_key = keys[frame_index]
            
            # Determine the title string dynamically for the current frame
            frame_title = title_dict[current_key] if title_dict else f"Key: {current_key:.4f}"
            ax.set_title(frame_title)
            return [im]
            
        # 4. Construct the unique animation object
        ani = FuncAnimation(
            fig, update, frames=len(frames), interval=interval, blit=True
        )
        
        # 5. Handle automatic file renaming if saving is requested
        if save_path:
            path = Path(save_path)
            start_str = str(start).replace('.', '_')
            end_str = str(end).replace('.', '_')
            
            # Construct new filename structure: parent_dir / "filename_start_to_end.ext"
            subset_path = path.parent / f"{path.stem}_{start_str}_to_{end_str}{path.suffix}"
            
            fps = int(1000 / interval)
            file_extension = subset_path.suffix.lower()
            
            if file_extension == '.gif':
                print(f"Saving range [{start}, {end}] to {subset_path.name}...")
                ani.save(subset_path, writer='pillow', fps=fps)
            elif file_extension == '.mp4':
                print(f"Saving range [{start}, {end}] to {subset_path.name}...")
                ani.save(subset_path, writer='ffmpeg', fps=fps, bitrate=1800)
            else:
                print(f"Warning: Extension '{file_extension}' not recognized. Save skipped.")
        
        # 6. Close the background canvas to prevent duplicate static image renders
        plt.close(fig)
        
        # 7. Generate HTML player widget, print a header, and push it live into the notebook
        html_player = HTML(ani.to_jshtml())
        print(f"\n=== Animation Range: {start} to {end} ===")
        display(html_player)
        
        animation_widgets.append(html_player)
        
    return animation_widgets

In [ ]:
# need to adjust to your current path used for your experiment
# adjust the h5_pattern as required for your experiment
h5_loc=Path(r'X:\bl17-2\2026_06_Lindenberg\BTO_001')
h5_pattern = 'run*data*.h5'
h5_files=h5_loc.glob(h5_pattern)
# print(list(h5_files))
intensities_on=OrderedDict()
intensities_off=OrderedDict()
for file in h5_files:
    # print(file)
    run=int(file.stem.split('_')[0].split('run')[-1])
    if run<3 or run>15:continue # could also do something like: if run<75:continue, this will plot runs greater than 75
    # print(file.stem.split('_')[4])
    voltage=file.stem.split('V')[0].split('_')[-1]
    if 'm' in voltage:
        voltage=float(voltage.split('m')[0])*1e-3
    delay=file.stem.split('_')[-4].split('delay')[-1].split('ns')[0]
    h5=h5py.File(file,'r')
    # print(h5['entry'])
    # h5.visititems(print)
    data_1=h5['entry/data/data'][0,200:300,500:750]
    data_off=h5['entry/data/data'][1,200:300,500:750]
    h5.close()
    data_on=np.where(data_on>4e8,np.nan, data_on) # removing hot pixels
    data_off=np.where(data_off>4e8,np.nan, data_off)
    intensity_on=np.nansum(data_on)
    intensity_off=np.nansum(data_off)
    intensities_on[voltage] = intensity_on
    intensities_off[voltage] = intensity_off
    # print(f'File: {file.stem}, Voltage: {voltage} V, Intensity On: {intensity_on}, Intensity Off: {intensity_off}')
    fig,ax=plt.subplots(1,3)
    im=ax[0].imshow(data_on,norm=colors.LogNorm(vmin=1e0,vmax=1e2))
    # im=ax[1].imshow(data_off,norm=colors.LogNorm(vmin=1e0,vmax=1e2))
    im=ax[1].imshow(data_on-data_off,norm=colors.Normalize(vmin=-10,vmax=10),cmap='bwr')
    plt.colorbar(im, ax=ax[1])
    im=ax[2].imshow((data_on-data_off)/data_off,norm=colors.Normalize(vmin=-1,vmax=1),cmap='bwr')
    plt.colorbar(im, ax=ax[2])
    fig.suptitle(f'run: {run}, Voltage: {voltage} V,delay: {delay} ns')
    fig,ax=plt.subplots(1,2)
    ax[0].plot(data_on[75,60:150],label='gate1')
    ax[0].plot(data_off[75,60:150],label='gate2')
    ax[0].legend()
    ax[1].plot((data_on[75,60:150]-data_off[75,60:150])/data_on[75,60:150],label='On-Off')

In [ ]:
#to plot an eta scan:
data_file=h5_loc / 'run82_BTO_001_peak002_static-eta-scan_scan001_data_000001.h5'
h5=h5py.File(data_file,'r')
data=h5['entry/data/data'][:,200:350,:]
data=np.where(data>4e8,np.nan, data)
h5.close()
etas=np.linspace(11.34,12.19,86)
fig,ax=plt.subplots(figsize=(10,5))
ax.imshow(data[data.shape[0]//2],norm=colors.LogNorm(vmin=1e0,vmax=1e4))
intensities=np.nansum(data,axis=(1,2))
print(*zip(etas,intensities/intensities.max()))
print(intensities.shape)
fig,ax=plt.subplots()
ax.plot(etas,intensities,'.-')
ax.set_xlabel('Eta')
ax.set_ylabel('Intensity (sum of pixel values)')

In [1]:
# to help determine static eta scan values:
#example for 002 peak bto
np.linspace(11.34,12.19,41)

In [2]:
#plotting a voltage driven eta scan:
h5_pattern = 'run*eta*data*.h5'
h5_files=h5_loc.glob(h5_pattern)
# print(list(h5_files))
intensities_on=OrderedDict()
intensities_off=OrderedDict()
diff_stack=OrderedDict()
norm_diff_stack=OrderedDict()
for file in h5_files:
    # print(file)
    run=int(file.stem.split('_')[0].split('run')[-1])
    if run<83 or run==107:continue
    eta_str=file.stem.split('eta')[-1].split('_')[0]
    eta_str=eta_str.replace('p','.')
    eta=float(eta_str)
    h5=h5py.File(file,'r')
    # print(h5['entry'])
    # h5.visititems(print)
    data_1=h5['entry/data/data'][0,200:350,:]
    data_2=h5['entry/data/data'][1,200:350,:]
    h5.close()
    data_1=np.where(data_1>1e4,np.nan, data_1)
    data_1_linout=np.nansum(data_1,axis=0)
    data_2=np.where(data_2>4e8,np.nan, data_2)
    data_2_linout=np.nansum(data_2,axis=0)
    diff_stack[eta]=data_2-data_1
    norm_diff_stack[eta]=(data_2-data_1)/data_1
    intensity_1=np.nansum(data_1)
    intensity_2=np.nansum(data_2)
    intensities_on[eta] = intensity_1
    intensities_off[eta] = intensity_2
    # print(f'File: {file.stem}, Voltage: {voltage} V, Intensity On: {intensity_on}, Intensity Off: {intensity_off}')
    fig,ax=plt.subplots(1,3,figsize=(20,5))
    # im=ax.imshow(data_1,norm=colors.LogNorm())
    im=ax[0].imshow(data_1,norm=colors.LogNorm())
    im=ax[1].imshow(data_2,norm=colors.LogNorm(vmin=1e0,vmax=1e2))
    im=ax[1].imshow(data_2-data_1,norm=colors.Normalize(vmin=-200,vmax=200),cmap='bwr')
    plt.colorbar(im, ax=ax[1])
    im=ax[2].imshow((data_2-data_1)/data_1,norm=colors.Normalize(vmin=-1,vmax=1),cmap='bwr')
    plt.colorbar(im, ax=ax[2])
    fig.suptitle(f'run: {run}, eta: {eta} deg')
    fig,ax=plt.subplots(1,2)
    ax[0].semilogy(data_1_linout,label='gate1')
    ax[0].semilogy(data_2_linout,label='gate2')
    ax[0].legend()
    ax[1].plot((data_2_linout-data_1_linout)/data_1_linout,label='On-Off')
    # ax[1].set_ylim(-0.1,1.1)

In [ ]:
# creating a movie
# 1. Sort the OrderedDict items by their float keys
sorted_items = sorted(diff_stack.items())
keys = [item[0] for item in sorted_items]
frames = [item[1] for item in sorted_items]

# 2. Set up the Matplotlib figure and axis
fig, ax = plt.subplots()
ax.axis('off')  # Turn off axis borders and ticks for a clean visual

# 3. Initialize the plot with the first image frame
# Note: If your images are grayscale, you can add cmap='gray' inside imshow()
im = ax.imshow(frames[0],norm=colors.Normalize(vmin=-1000,vmax=1000),cmap='bwr')
ax.set_title(f"Key: {keys[0]:.4f}")

# 4. Define the animation update logic
def update(frame_index):
    # Update image data
    im.set_data(frames[frame_index])
    # Update title dynamically with the sorted float key
    ax.set_title(f"Key: {keys[frame_index]:.4f}")
    return [im]

# 5. Build the animation object
ani = FuncAnimation(
    fig, 
    update, 
    frames=len(frames), 
    interval=200,  # Speed of playback: 200ms per frame (5 FPS)
    blit=True
)

# 6. Close the background figure window so VS Code doesn't draw a duplicate static image
plt.close(fig)

# 7. Render the interactive JavaScript player right inside the notebook cell
HTML(ani.to_jshtml())

In [ ]:
#plotting a delay scan intensity vs delay time, can adjust to include other moments
h5_pattern = 'run*data*.h5'
h5_files=h5_loc.glob(h5_pattern)
# print(list(h5_files))
intensities_on=OrderedDict()
intensities_off=OrderedDict()
diff_stack=OrderedDict()
norm_diff_stack=OrderedDict()
delay_stack=OrderedDict()
for file in h5_files:
    # print(file)
    run=int(file.stem.split('_')[0].split('run')[-1])
    if run < 178 or run > 203:continue
    # print(file)
    # print(file.stem.split('_')[4])
    voltage=file.stem.split('V')[0].split('_')[-1]
    if 'm' in voltage:
        voltage=float(voltage.split('m')[0])*1e-3
    delay=file.stem.split('delay')[-1].split('_')[0].split('us')[0]
    # print(delay)
    delay=float(delay.replace('p','.'))
    h5=h5py.File(file,'r')
    # print(h5['entry'])
    # h5.visititems(print)
    data_1=h5['entry/data/data'][0,250:300,580:635]
    data_2=h5['entry/data/data'][1,250:300,580:635]
    h5.close()
    data_1=np.where(data_1>4e8,np.nan, data_1)
    data_2=np.where(data_2>4e8,np.nan, data_2)
    diff_stack[run]=data_2-data_1
    norm_diff_stack[run]=(data_2-data_1)/data_1
    intensity_on=np.nansum(data_1)
    intensity_off=np.nansum(data_2)
    intensities_on[run] = intensity_on
    intensities_off[run] = intensity_off
    delay_stack[run] = delay
    print(f'File: {file.stem}, Delay: {delay} us, Intensity On: {intensity_on}, Intensity Off: {intensity_off}')
    # fig,ax=plt.subplots(1,3)
    # im=ax[0].imshow(data_1,norm=colors.LogNorm(vmin=1e0,vmax=1e2))
    # # im=ax[1].imshow(data_2,norm=colors.LogNorm(vmin=1e0,vmax=1e2))
    # im=ax[1].imshow(data_1-data_2,norm=colors.Normalize(vmin=-10,vmax=10),cmap='bwr')
    # plt.colorbar(im, ax=ax[1])
    # im=ax[2].imshow((data_1-data_2)/data_2,norm=colors.Normalize(vmin=-1,vmax=1),cmap='bwr')
    # plt.colorbar(im, ax=ax[2])
    # fig.suptitle(f'run: {run}, Voltage: {voltage} V, Delay: {delay} us')
    # fig,ax=plt.subplots(1,2)
    # ax[0].plot(data_1[75,60:150],label='gate1')
    # ax[0].plot(data_2[75,60:150],label='gate2')
    # ax[0].legend()
    # ax[1].plot((data_1[75,60:150]-data_2[75,60:150])/data_1[75,60:150],label='On-Off')

In [ ]:
# movie of time delay scan
animations = animate_image_subsets(diff_stack, title_dict=delay_stack, interval=200)#, save_path=r'X:\bl17-2\2026_06_Lindenberg\analysis\plots\runs126-147_delay-scan_diff-images.gif')

In [ ]:
peak_diff_stack=OrderedDict()
for key, value in diff_stack.items():
    # fig,ax=plt.subplots()
    # ax.plot(np.nansum(value,axis=0))
    peak_diff_stack[key] = np.nansum(value[:,:30])-np.nansum(value[:,30:60])

fig,ax=plt.subplots()
ax.plot(delay_stack.values(), list(peak_diff_stack.values()), linestyle='-',marker='o')
ax.set_xlabel('Delay (us)')
ax.set_ylabel('Peak Difference (sum of first 30 columns \n - sum of next 30 columns)')